# Can we predict `passed_quiz`?
Follows `checklist.md`. Key facts about the data: no date column (row order is not treated as time), and 1,800 students contribute ~7 rows each, so every split holds **whole students** (grouped).

**Feature timing (checklist 7).** `student_key` is an identifier and is dropped. The target is the quiz closing a session, so I assume the prediction is made when the quiz starts: `session_minutes` and `hints_used` (accumulated during tutoring) are treated as known then. If the model must predict at session start, those two are not available — the `logreg_pre_session` candidate below covers that case. `prior_score` is described as a placement-test score 'as stored on the session record' but varies across a student's sessions, so its provenance is unclear; it is kept but flagged.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, log_loss
df = pd.read_csv('data.csv')
print(df.shape, 'students:', df.student_key.nunique(), 'pass rate: %.3f' % df.passed_quiz.mean())
print(df.groupby('student_key').prior_score.nunique().describe()[['mean','max']])

## 1. Test set first (whole students)
Nothing below looks at the test rows until the final step.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert not set(train.student_key) & set(test.student_key)
print(len(train), len(test))
y_tr, g_tr = train.passed_quiz.values, train.student_key.values

## 2. Candidates and plan (named before any scoring)
- `baseline`: predicts the training pass rate (trivial).
- `logreg_all`: standardised logistic regression on enrol_year, prior_score, session_minutes, hints_used (school_code is a nominal code, so it is not used linearly).
- `gbm_all_d2`, `gbm_all_d3`: histogram gradient boosting on those four plus school_code, max_depth 2 or 3.
- `logreg_pre_session`: logistic regression on enrol_year, prior_score only (what is known before the session).

Selection metric: CV ROC AUC (threshold-free). Decision threshold: chosen on CV out-of-fold predictions to maximise F1. 5-fold grouped CV inside train; preprocessing lives inside pipelines so it is fit per fold.

In [ ]:
F_ALL = ['enrol_year','prior_score','session_minutes','hints_used']
F_GBM = F_ALL + ['school_code']
F_PRE = ['enrol_year','prior_score']
def lr(cols): return make_pipeline(ColumnTransformer([('s',StandardScaler(),cols)]), LogisticRegression(max_iter=1000))
def gb(cols,d): return make_pipeline(ColumnTransformer([('p','passthrough',cols)]), HistGradientBoostingClassifier(max_depth=d, random_state=0))
cands = {
 'baseline': (DummyClassifier(strategy='prior'), F_ALL),
 'logreg_all': (lr(F_ALL), F_ALL),
 'gbm_all_d2': (gb(F_GBM,2), F_GBM),
 'gbm_all_d3': (gb(F_GBM,3), F_GBM),
 'logreg_pre_session': (lr(F_PRE), F_PRE),
}
cv = GroupKFold(n_splits=5)
oof, rows = {}, []
for n,(m,cols) in cands.items():
    p = cross_val_predict(m, train[cols], y_tr, groups=g_tr, cv=cv, method='predict_proba')[:,1]
    oof[n]=p
    rows.append((n, roc_auc_score(y_tr,p), log_loss(y_tr,p)))
cvres = pd.DataFrame(rows, columns=['model','cv_auc','cv_logloss']).set_index('model'); cvres

In [ ]:
best = cvres.cv_auc.idxmax()
ths = np.linspace(0.05,0.95,91)
f1s = [f1_score(y_tr, oof[best]>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold: %.2f' % thr, '| CV F1 at threshold: %.3f' % max(f1s))

## 3. Decision (fixed before touching the test set)
The model with the highest CV AUC and the F1-optimal CV threshold printed above are the final choice. The test set is scored once, next cell.

In [ ]:
m, cols = cands[best]
m.fit(train[cols], y_tr)
p = m.predict_proba(test[cols])[:,1]; yt = test.passed_quiz.values
print('HEADLINE test AUC: %.3f | accuracy: %.3f | F1: %.3f (threshold %.2f)' % (roc_auc_score(yt,p), accuracy_score(yt,p>=thr), f1_score(yt,p>=thr), thr))
print('always-predict-majority accuracy: %.3f' % max(yt.mean(),1-yt.mean()))
rng=np.random.default_rng(0); s=test.student_key.values; ids=np.unique(s); idx={i:np.where(s==i)[0] for i in ids}
aucs=[roc_auc_score(yt[j],p[j]) for j in (np.concatenate([idx[i] for i in rng.choice(ids,len(ids))]) for _ in range(500))]
print('95%% student-bootstrap CI for AUC: %.3f-%.3f' % tuple(np.percentile(aucs,[2.5,97.5])))

## Conclusions
Interpret the headline against the baseline (AUC 0.5) and the CV table. Caveats: (1) no timestamps, so this measures generalisation to unseen students, not to future sessions; (2) the timing of `session_minutes`/`hints_used` and the meaning of `prior_score` should be confirmed with the data owner; (3) the test set was used once.